In [99]:
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import classification_report, multilabel_confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
import copy

# OpenMP 충돌 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [100]:
Config = {
    "NUM_CLASSES": 5,         # 0 normal, 1 dos, 2 fuzzing, 3 replay, 4 spoofing
    "BATCH_SIZE": 64,
}

In [101]:
# ================================
# 1. Causal Convolution 레이어 정의
# ================================
class CausalConv1d(nn.Module):
    """
    미래의 데이터를 보지 않도록 왼쪽으로만 패딩을 넣는 컨볼루션 레이어입니다.
    """
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super(CausalConv1d, self).__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.padding, dilation=dilation)

    def forward(self, x):
        x = self.conv(x)
        # 오른쪽 끝부분(미래 데이터 위치)을 잘라내어 인과성을 유지합니다.
        if self.padding != 0:
            x = x[:, :, :-self.padding]
        return x

In [102]:
# ================================
# 2. SeqIDS 모델 (일반화 기법 적용: Dropout + BatchNorm)
# ================================
class SeqIDS(nn.Module):
    def __init__(self, num_classes=5, dropout_rate=0.5):
        super(SeqIDS, self).__init__()
        
        # Layer 1: Conv -> BN -> ReLU -> Dropout
        self.layer1 = CausalConv1d(9, 32, kernel_size=5)
        self.bn1 = nn.BatchNorm1d(32)     # [일반화] 배치 정규화
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=dropout_rate) # [일반화] 드롭아웃
        
        # Layer 2: Conv -> BN -> ReLU -> Dropout
        self.layer2 = CausalConv1d(32, 64, kernel_size=3)
        self.bn2 = nn.BatchNorm1d(64)     # [일반화] 배치 정규화
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=dropout_rate) # [일반화] 드롭아웃
        
        # Classifier
        self.classifier = nn.Conv1d(64, num_classes, kernel_size=1)

    def forward(self, x):
        # x shape: (Batch, 9, 64)
        x = self.layer1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.layer2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        logits = self.classifier(x)  # (Batch, 5, 64)
        return logits

In [103]:
# ================================
# 3. 데이터셋 클래스
# ================================
class SeqWindowDataset(Dataset):
    def __init__(self, tensor_X, tensor_y):
        # [수정] NumPy 배열이 들어올 경우를 대비한 변환 로직
        if isinstance(tensor_X, np.ndarray):
            tensor_X = torch.from_numpy(tensor_X).float()
        if isinstance(tensor_y, np.ndarray):
            tensor_y = torch.from_numpy(tensor_y).long()

        # [추가] NaN(결측치)을 0.0으로 안전하게 대체하여 학습 붕괴 방지
        self.X = torch.nan_to_num(tensor_X, nan=0.0)
        self.y = tensor_y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [104]:
# ================================
# 4. 평가 함수 (Window Level Evaluation)
# ================================
def evaluate_packet_level_multilabel(model, loader, device):
    model.eval()
    all_preds, all_targets = [], []
    # 라벨 이름 (인코딩 시 설정한 순서대로)
    target_names = ['Normal', 'DoS', 'Fuzzing', 'Replay', 'Spoofing']
    mlb = MultiLabelBinarizer(classes=[0, 1, 2, 3, 4])

    print("\n🔍 평가 진행 중...")
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            logits = model(X_batch)
            
            # (Batch, 64) - 각 패킷별 예측 클래스
            preds = logits.argmax(dim=1)

            # 2. 모든 패킷을 1차원으로 펼침 (Flatten)
            # numpy를 사용하면 리스트 comprehension보다 훨씬 빠릅니다.
            p_flat = preds.cpu().numpy().flatten()
            t_flat = y_batch.cpu().numpy().flatten()
            
            # Multi-label 평가 포맷에 맞춰 리스트화
            all_preds.extend([[p] for p in p_flat])
            all_targets.extend([[t] for t in t_flat])

    # 3. 이진 변환 및 지표 계산
    y_true = mlb.fit_transform(all_targets)
    y_pred = mlb.transform(all_preds)

    print("\n" + "="*60)
    print("📊 [Packet-Level] Multi-Label Evaluation Results")
    print("   (Total Packets Evaluated: {:,})".format(len(all_targets)))
    print("="*60)
    # 이제 Support 숫자가 윈도우 개수가 아닌 전체 패킷 개수로 나올 것입니다.
    print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))
    
    return multilabel_confusion_matrix(y_true, y_pred)

In [105]:
# ================================
# 5. 학습 함수 (일반화 5대장 적용 완료)
# ================================
def train_seqids_advanced(dataset_npz_path, epochs, lr=1e-4, device=None, patience=3):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🖥 실행 디바이스: {device}")

    # 1. 데이터 로드
    print(f"📂 데이터 로딩 중: {dataset_npz_path}")
    data_np = np.load(dataset_npz_path)
    X_np, y_np = data_np["X"], data_np["y"]

    # NaN이 X의 어느 feature 중에서 가장 많이 나오는지 확인(인코딩 잘못됐는지 확인)
    for i in range(X_np.shape[1]):
        feat_nan = np.isnan(X_np[:, i, :]).sum()
        print(f"Feature {i}의 NaN 개수: {feat_nan}")

    
    nan_count_X = np.isnan(X_np).sum()
    nan_count_y = np.isnan(y_np).sum()
    print(f"🔍 데이터 검사 결과:")
    print(f"   - X 내 NaN 개수: {nan_count_X}개")
    print(f"   - y 내 NaN 개수: {nan_count_y}개")
    if nan_count_X > 0:
        print("   ⚠️ 주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.")


    inf_count = np.isinf(X_np).sum() # 무한대 체크 추가
    print(f"🔍 X 내 inf 개수: {inf_count}개")
    if inf_count > 0:
        print("⚠️ 경고: 데이터에 inf(무한대)가 포함되어 있습니다. 인코딩 스크립트를 수정하세요.")


    full_dataset = SeqWindowDataset(X_np, y_np)

    # [일반화 1] Validation Split (80:20)
    total_size = len(full_dataset)
    train_size = int(0.8 * total_size)
    val_size = total_size - train_size
    
    # 시드 고정 (재현성을 위해)
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], generator=generator)
    
    print(f"📊 데이터 분할 완료: 학습 {train_size}개 / 검증 {val_size}개")
    
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

    # 2. 모델 및 옵티마이저
    # [일반화 2] Dropout 적용 (0.5)
    model = SeqIDS(num_classes=5, dropout_rate=0.5).to(device)
    
    # [일반화 3] Weight Decay (L2 Regularization) 적용 -> 1e-4
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-2)
    # [일반화 4] Learning Rate Scheduler (성능 정체 시 학습률 감소)
    # 3번(patience) 동안 Val Loss가 안 줄어들면 학습률을 반토막(0.5) 냄
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    # 불균형 데이터 가중치 (공격 클래스 중요도 Up)
    weights = torch.tensor([1.0, 2.0, 2.0, 2.0, 2.0]).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # [일반화 5] Early Stopping 변수
    best_val_loss = float('inf')
    best_model_state = None
    patience_check = 0

    print("\n🚀 학습 시작 (Advanced Mode)...")
    
    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            logits = model(X_batch)
            # (Batch, Class, Time) -> (Batch*Time, Class) 형태로 변형하여 Loss 계산
            loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5), y_batch.reshape(-1))
            
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                
                logits = model(X_val)
                loss = criterion(logits.permute(0, 2, 1).reshape(-1, 5), y_val.reshape(-1))
                val_loss += loss.item()
        
        avg_val_loss = val_loss / len(val_loader)
        
        # 스케줄러 업데이트 (Val Loss 기준)
        scheduler.step(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch [{epoch}/{epochs}] Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f} | LR: {current_lr:.6f}")

        # --- Early Stopping Logic ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict()) # 모델 가중치 백업
            patience_check = 0 # 카운트 초기화
            # print("  ✅ Best Model 갱신!")
        else:
            patience_check += 1
            print(f"  ⚠️ 검증 손실 개선 안됨 (Patience: {patience_check}/{patience})")
            if patience_check >= patience:
                print(f"🛑 Early Stopping 발동! 학습을 조기 종료합니다.")
                break

    # 학습 종료 후, 가장 좋았던 모델 상태로 복구
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("\n💾 검증 성능이 가장 좋았던(Best) 모델로 복구되었습니다.")

    # 검증 로더도 반환하여 최종 평가에 사용
    return model, val_loader, device

In [106]:
# ================================
# 6. 메인 실행 블록
# ================================
if __name__ == "__main__":
    # [주의] 본인의 실제 파일 경로로 수정하세요
    DATA_PATH = "C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/test_dataset_all_CS.npz"
    
    # 1) 향상된 학습 진행 (Validation, Dropout, BatchNorm, Scheduler 등 모두 적용)
    # 리턴받는 val_loader는 학습에 쓰지 않은 '순수 검증용' 데이터입니다.
    trained_model, val_loader, device = train_seqids_advanced(DATA_PATH, epochs=20, patience=7)

    # 2) 검증 데이터셋(20%)에 대한 최종 상세 평가
    print("\n🔍 최종 검증 데이터셋(Validation Set) 평가 결과:")
    evaluate_packet_level_multilabel(trained_model, val_loader, device)

    # 3) 모델 저장
    MODEL_SAVE_PATH = "C:/Users/user/Desktop/IDS_masters/재호/model4.pth"
    torch.save(trained_model.state_dict(), MODEL_SAVE_PATH)
    print(f"\n💾 Best 모델 저장이 완료되었습니다: {MODEL_SAVE_PATH}")

🖥 실행 디바이스: cuda
📂 데이터 로딩 중: C:/Users/user/Desktop/IDS_masters/small_CAN_MIRGU/test_dataset_all_CS.npz
Feature 0의 NaN 개수: 0
Feature 1의 NaN 개수: 162
Feature 2의 NaN 개수: 0
Feature 3의 NaN 개수: 0
Feature 4의 NaN 개수: 0
Feature 5의 NaN 개수: 113596
Feature 6의 NaN 개수: 0
Feature 7의 NaN 개수: 0
Feature 8의 NaN 개수: 0
🔍 데이터 검사 결과:
   - X 내 NaN 개수: 113758개
   - y 내 NaN 개수: 0개
   ⚠️ 주의: 입력 데이터(X)의 NaN은 0.0으로 자동 대체되어 학습됩니다.
🔍 X 내 inf 개수: 0개
📊 데이터 분할 완료: 학습 93800개 / 검증 23450개

🚀 학습 시작 (Advanced Mode)...
Epoch [1/20] Train Loss: 0.63621 | Val Loss: 0.27949 | LR: 0.000100
Epoch [2/20] Train Loss: 0.25466 | Val Loss: 0.17188 | LR: 0.000100
Epoch [3/20] Train Loss: 0.18371 | Val Loss: 0.14993 | LR: 0.000100
Epoch [4/20] Train Loss: 0.16297 | Val Loss: 0.16225 | LR: 0.000100
  ⚠️ 검증 손실 개선 안됨 (Patience: 1/7)
Epoch [5/20] Train Loss: 0.15471 | Val Loss: 0.21024 | LR: 0.000100
  ⚠️ 검증 손실 개선 안됨 (Patience: 2/7)
Epoch [6/20] Train Loss: 0.15234 | Val Loss: 0.25462 | LR: 0.000100
  ⚠️ 검증 손실 개선 안됨 (Patience: 3/7)
Epoch [7/2